# LLM judge로 궤적 평가

이 노트북은 Python 3.12 새 커널에서 독립 실행합니다.

## 패키지 설치

In [1]:
# 노트북이 사용하는 패키지 버전을 새 커널 환경에 설치합니다.
%pip install -q agentevals==0.0.9 langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 환경 준비


In [2]:
# 공개하지 않은 API 키는 존재 여부만 확인하고 고정한 패키지 버전을 출력합니다.
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
print("agentevals", version("agentevals"))
print("langchain", version("langchain"))
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."

agentevals 0.0.9
langchain 1.4.0


## 평가할 궤적

### 코드 읽기와 비교 경계

`create_trajectory_llm_as_judge(...)`는 평가용 모델과 루브릭 프롬프트를 묶은 평가 함수를 만듭니다. `outputs=actual`과 `reference_outputs=reference`를 전달하면 두 메시지 궤적을 텍스트로 구성해 실제 모델을 호출하고, 점수와 설명을 사전으로 반환합니다. 여기서 judge는 피평가 에이전트와 별개의 모델 호출입니다. 고정된 `actual`을 평가하더라도 judge의 문장이나 점수는 모델과 프롬프트에 따라 달라질 수 있습니다.

결정적 매칭과 judge는 역할이 다릅니다. 필수 도구 이름과 순서처럼 코드로 정확히 표현할 수 있는 규칙은 결정적 검사로 빠르고 재현 가능하게 확인합니다. 여러 합리적 경로의 의미나 불필요한 단계의 정도는 judge가 읽을 수 있지만, 루브릭에 없는 보안 규칙을 자동으로 발견한다고 기대해서는 안 됩니다. 이 예제는 실제와 기준을 동일하게 만들었으므로 두 평가가 모두 통과하는 기본 동작만 확인합니다. 다른 경로의 의미 차이를 비교하려면 별도의 대조 궤적이 필요합니다.


In [3]:
# judge와 결정적 평가가 같은 실제·기준 메시지를 읽도록 입력을 고정합니다.
from langchain.messages import HumanMessage, AIMessage, ToolMessage
def trace(*tools: str) -> list:
    messages=[HumanMessage("규정을 확인해 주세요.")]
    for i,name in enumerate(tools,1):
        messages += [AIMessage("", tool_calls=[{"name":name,"args":{"topic":"대출"},"id":f"c{i}","type":"tool_call"}]), ToolMessage("확인", tool_call_id=f"c{i}")]
    messages.append(AIMessage("일반 도서는 14일입니다."))
    return messages
reference=trace("search_rules", "check_source")
actual=trace("search_rules","check_source")

## 실제 LLM judge 실행

이 셀에서만 judge 모델을 실제로 호출합니다. 반환 사전의 `score`와 `comment`를 함께 저장해 판정과 근거를 같이 검토합니다.


In [4]:
# TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE로 전체 단계의 논리와 효율을 실제 모델이 판정합니다.
from agentevals.trajectory.llm import create_trajectory_llm_as_judge, TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE
evaluator=create_trajectory_llm_as_judge(model="openai:gpt-5.4-mini",prompt=TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE)
evaluation=evaluator(outputs=actual,reference_outputs=reference)
print(evaluation)
assert evaluation["score"] in [True, False]

{'key': 'trajectory_accuracy', 'score': True, 'comment': 'The actual trajectory matches the reference trajectory exactly in sequence, tool usage, and final response. It makes logical sense and is semantically equivalent to the reference. Thus, the score should be: true.', 'metadata': None}


## 결정적 검사와 함께 보기

권한, 금지 도구, 필수 순서처럼 코드로 정확히 표현할 수 있는 규칙은 결정적 검사로 남깁니다. judge는 의미 판단이 필요한 범위를 보완합니다.


In [5]:
# 같은 궤적의 strict 점수와 judge 설명을 나란히 출력해 구조와 의미 평가를 구분합니다.
from agentevals.trajectory.match import create_trajectory_match_evaluator
deterministic=create_trajectory_match_evaluator(trajectory_match_mode="strict")(outputs=actual,reference_outputs=reference)
print("strict:", deterministic)
print("judge:", evaluation)
assert deterministic["score"] is True

strict: {'key': 'trajectory_strict_match', 'score': True, 'comment': None, 'metadata': None}
judge: {'key': 'trajectory_accuracy', 'score': True, 'comment': 'The actual trajectory matches the reference trajectory exactly in sequence, tool usage, and final response. It makes logical sense and is semantically equivalent to the reference. Thus, the score should be: true.', 'metadata': None}
